In [19]:
import glob
import os
import re
import pandas as pd

data_dir = os.path.join("..", "data")

# Panel year ("Anno") = population reference year (1 January of that year).
# Income of tax year T is attached to panel year T + INCOME_LAG (income 2020 -> Anno 2021, ..., 2024 -> 2025).
INCOME_LAG = 1

# ---------- ISTAT population: one file per province per reference year ----------
def load_posas(path):
    """Read a POSAS csv; finds the header row instead of assuming skiprows=1."""
    with open(path, encoding="utf-8-sig") as f:
        skip = next(i for i, line in enumerate(f) if "Codice comune" in line)
    df = pd.read_csv(path, sep=";", skiprows=skip, encoding="utf-8-sig",
                     dtype={"Codice comune": str})
    df = df[df["Codice comune"].str.fullmatch(r"\d{6}", na=False)].copy()
    df["Anno"] = int(re.search(r"POSAS_(\d{4})_", os.path.basename(path)).group(1))
    return df

posas_files = sorted(glob.glob(os.path.join(data_dir, "POSAS_*_it_05[45]_*.csv")))
umbria_demographics = pd.concat([load_posas(p) for p in posas_files], ignore_index=True)
umbria_demographics["Codice_Comune"] = umbria_demographics["Codice comune"].astype(int)

# Prendi le chiavi dei 92 comuni sicuri dell'Umbria
comuni_keys = umbria_demographics["Codice_Comune"].unique()


# ---------- MEF income: explicit loading for all 5 years (2020-2024) ----------
# ---------- MEF income: explicit loading for all 5 years (2020-2024) ----------
income_filenames = [
    "Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2020.csv",
    "Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2021.csv",
    "Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2022.csv",
    "Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2023.csv",
    "Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2024.csv"
]

print(income_filenames)

# Forza Pandas a mostrare tutte le colonne senza i "..."
pd.set_option('display.max_columns', None)

income_frames = []
for fname in income_filenames:
    path = os.path.join(data_dir, fname)
    if os.path.exists(path):
        df = pd.read_csv(path, sep=";", encoding="utf-8-sig", dtype=str,
                         index_col=False, usecols=lambda column: True)
        
        # 1. FIX CRITICO: Rimuove gli spazi iniziali e finali invisibili da tutti i nomi delle colonne
        df.columns = df.columns.str.strip()
        
        # Filtro sicuro basato sulla provincia
        if "Sigla Provincia" in df.columns:
            df_umbria = df[df["Sigla Provincia"].isin(["PG", "TR"])].copy()
        else:
            df["Codice Istat Comune"] = pd.to_numeric(df["Codice Istat Comune"], errors="coerce")
            df_umbria = df[df["Codice Istat Comune"].between(54000, 55999)].copy()
            
        # ---------------- ESTRAZIONE REDDITO A FASCE ---------------- #
        cols_to_clean = [c for c in df_umbria.columns if any(k in c for k in ['Ammontare in euro', 'Frequenza', 'contribuenti'])]
        
        for c in cols_to_clean:
            df_umbria[c] = pd.to_numeric(
                df_umbria[c].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False), 
                errors='coerce'
            ).fillna(0)
            
        bracket_amount_cols = [c for c in df_umbria.columns if 'Reddito complessivo' in c and 'Ammontare in euro' in c and '-' in c]
        
        if bracket_amount_cols:
            df_umbria['Reddito complessivo - Ammontare in euro_CALCOLATO'] = df_umbria[bracket_amount_cols].sum(axis=1)
        # ------------------------------------------------------------ #
        
        income_frames.append(df_umbria)
    else:
        print(f"Attenzione: file non trovato -> {fname}")

umbria_income = pd.concat(income_frames, ignore_index=True)
# Pulizia finale e conversione codici
umbria_income["Codice Istat Comune"] = umbria_income["Codice Istat Comune"].astype(str).str.strip()
umbria_income["Codice_Comune"] = pd.to_numeric(umbria_income["Codice Istat Comune"], errors="coerce").astype(int)
umbria_income["Anno di imposta"] = pd.to_numeric(umbria_income["Anno di imposta"])
umbria_income["Anno"] = umbria_income["Anno di imposta"] + INCOME_LAG

# ---------- Verifiche finali ----------
print("Population files count:", len(posas_files))
print("Population rows per year:\n", umbria_demographics.groupby("Anno")["Codice_Comune"].nunique().to_string())
print("\nIncome rows per tax year:\n", umbria_income.groupby("Anno di imposta")["Codice_Comune"].nunique().to_string())
display(umbria_income.head(10))

['Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2020.csv', 'Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2021.csv', 'Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2022.csv', 'Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2023.csv', 'Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2024.csv']
Population files count: 12
Population rows per year:
 Anno
2021    92
2022    92
2023    92
2024    92
2025    92
2026    92

Income rows per tax year:
 Anno di imposta
2020    92
2021    92
2022    92
2023    92
2024    92


,Anno di imposta,Codice catastale,Codice Istat Comune,Denominazione Comune,Sigla Provincia,Regione,Codice Istat Regione,Numero contribuenti,Reddito da fabbricati - Frequenza,Reddito da fabbricati - Ammontare in euro,Reddito da lavoro dipendente e assimilati - Frequenza,Reddito da lavoro dipendente e assimilati - Ammontare in euro,Reddito da pensione - Frequenza,Reddito da pensione - Ammontare in euro,Reddito da lavoro autonomo (comprensivo dei valori nulli) - Frequenza,Reddito da lavoro autonomo (comprensivo dei valori nulli) - Ammontare in euro,Reddito di spettanza dell'imprenditore in contabilita' ordinaria (comprensivo dei valori nulli) - Frequenza,Reddito di spettanza dell'imprenditore in contabilita' ordinaria (comprensivo dei valori nulli) - Ammontare in euro,Reddito di spettanza dell'imprenditore in contabilita' semplificata (comprensivo dei valori nulli) - Frequenza,Reddito di spettanza dell'imprenditore in contabilita' semplificata (comprensivo dei valori nulli) - Ammontare in euro,Reddito da partecipazione (comprensivo dei valori nulli) - Frequenza,Reddito da partecipazione (comprensivo dei valori nulli) - Ammontare in euro,Reddito imponibile - Frequenza,Reddito imponibile - Ammontare in euro,Imposta netta - Frequenza,Imposta netta - Ammontare in euro,Bonus spettante - Frequenza,Bonus spettante - Ammontare in euro,Reddito imponibile addizionale - Frequenza,Reddito imponibile addizionale - Ammontare in euro,Addizionale regionale dovuta - Frequenza,Addizionale regionale dovuta - Ammontare in euro,Addizionale comunale dovuta - Frequenza,Addizionale comunale dovuta - Ammontare in euro,Reddito complessivo minore o uguale a zero euro - Frequenza,Reddito complessivo minore o uguale a zero euro - Ammontare in euro,Reddito complessivo da 0 a 10000 euro - Frequenza,Reddito complessivo da 0 a 10000 euro - Ammontare in euro,Reddito complessivo da 10000 a 15000 euro - Frequenza,Reddito complessivo da 10000 a 15000 euro - Ammontare in euro,Reddito complessivo da 15000 a 26000 euro - Frequenza,Reddito complessivo da 15000 a 26000 euro - Ammontare in euro,Reddito complessivo da 26000 a 55000 euro - Frequenza,Reddito complessivo da 26000 a 55000 euro - Ammontare in euro,Reddito complessivo da 55000 a 75000 euro - Frequenza,Reddito complessivo da 55000 a 75000 euro - Ammontare in euro,Reddito complessivo da 75000 a 120000 euro - Frequenza,Reddito complessivo da 75000 a 120000 euro - Ammontare in euro,Reddito complessivo oltre 120000 euro - Frequenza,Reddito complessivo oltre 120000 euro - Ammontare in euro,Reddito complessivo - Ammontare in euro_CALCOLATO,Trattamento spettante - Frequenza,Trattamento spettante - Ammontare in euro,Reddito complessivo - Frequenza,Reddito complessivo - Ammontare in euro,,Codice_Comune,Anno
0,2020,A045,055001,ACQUASPARTA,TR,Umbria,10,3167,1481,1286309,1520,27575629,1333,22619898,31.0,1207007.0,12.0,162904.0,102.0,1427140.0,173.0,1510318.0,3027,54413182,2378,9077835,896.0,385543.0,2336,50570761,2289,699549,2290.0,399131.0,0.0,0.0,877,4029804,472.0,5928170.0,1056,21406192,616,20662530,38.0,2417342.0,18.0,1656601.0,9.0,1362585.0,57463224.0,NaN,NaN,NaN,NaN,NaN,55001,2021
1,2020,A207,055002,ALLERONA,TR,Umbria,10,1288,629,407625,673,12143965,548,8308500,0.0,0.0,4.0,153395.0,39.0,419975.0,53.0,470868.0,1235,21284650,967,3371573,394.0,171417.0,951,19645579,928,269087,926.0,155539.0,0.0,0.0,358,1796104,227.0,2832369.0,441,8744040,219,7387056,9.0,557534.0,0.0,0.0,5.0,723216.0,22040319.0,NaN,NaN,NaN,NaN,NaN,55002,2021
2,2020,A242,055003,ALVIANO,TR,Umbria,10,1004,481,311742,474,8085330,431,7439632,5.0,97621.0,6.0,87270.0,38.0,396860.0,51.0,433031.0,970,16445315,726,2619484,239.0,102876.0,708,15063735,691,209096,661.0,89641.0,0.0,0.0,328,1529634,161.0,1981106.0,278,5741711,211,7076229,10.0,637818.0,0.0,0.0,0.0,0.0,16966498.0,NaN,NaN,NaN,NaN,NaN,55003,2021
3,2020,A262,055004,AMELIA,TR,Umbria,10,8359,4070,3985327,4006,77153067,3420,60751072,99.0,3799464.0,58.0,1034036.0,267.0,3795048.0,425.0,4369087.0,7934,150552563,614

In [20]:
import os
import pandas as pd

data_dir = os.path.join("..", "data")
file_2020 = os.path.join(data_dir, "Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2020.csv")

# Read just the first row to see how the Ministry named the columns in 2020
df_test = pd.read_csv(file_2020, sep=";", nrows=1, encoding="utf-8-sig")

print("Colonne in 2020:")
print(df_test.columns.tolist())

Colonne in 2020:
['Anno di imposta', 'Codice catastale', 'Codice Istat Comune', 'Denominazione Comune', 'Sigla Provincia', 'Regione', 'Codice Istat Regione', 'Numero contribuenti', 'Reddito da fabbricati - Frequenza', 'Reddito da fabbricati - Ammontare in euro', 'Reddito da lavoro dipendente e assimilati - Frequenza', 'Reddito da lavoro dipendente e assimilati - Ammontare in euro', 'Reddito da pensione - Frequenza', 'Reddito da pensione - Ammontare in euro', 'Reddito da lavoro autonomo (comprensivo dei valori nulli) - Frequenza', 'Reddito da lavoro autonomo (comprensivo dei valori nulli) - Ammontare in euro', "Reddito di spettanza dell'imprenditore in contabilita' ordinaria  (comprensivo dei valori nulli) - Frequenza", "Reddito di spettanza dell'imprenditore in contabilita' ordinaria  (comprensivo dei valori nulli) - Ammontare in euro", "Reddito di spettanza dell'imprenditore in contabilita' semplificata (comprensivo dei valori nulli) - Frequenza", "Reddito di spettanza dell'imprendito

In [21]:
# Standardize names of comuni (still needed for the OMI macro-area mapping below)
umbria_income['Comune_Key'] = umbria_income['Denominazione Comune'].astype(str).str.strip().str.upper()

# Keep only realistic ages (drops the 999 "total" row), then sum ages -> population per comune AND year
umbria_demographics['Età_Num'] = pd.to_numeric(umbria_demographics['Età'], errors='coerce')
umbria_filtered = umbria_demographics[umbria_demographics['Età_Num'] < 150]

istat_aggreg = (
    umbria_filtered.groupby(['Codice_Comune', 'Anno'], as_index=False)['Totale'].sum()
    .rename(columns={'Totale': 'Popolazione Totale'})
)

print("Anni disponibili in umbria_income:", sorted(umbria_income['Anno'].unique()))
print("Anni disponibili in istat_aggreg:", sorted(istat_aggreg['Anno'].unique()))

# Merge on ISTAT code + panel year (NOT on the comune name: "CITTÀ" vs "CITTA'" silently dropped 2 comuni)
dataset_completo = pd.merge(
    umbria_income, istat_aggreg,
    on=['Codice_Comune', 'Anno'], how='inner', validate='one_to_one'
)
dataset_completo = dataset_completo.sort_values(['Codice_Comune', 'Anno']).reset_index(drop=True)

# Verify merge: which years were dropped, and 92 comuni per remaining year
print("Anni popolazione scartati:", sorted(set(istat_aggreg['Anno']) - set(dataset_completo['Anno'])))
print("Anni reddito scartati:   ", sorted(set(umbria_income['Anno']) - set(dataset_completo['Anno'])))
print(f"Righe totali dopo il merge: {len(dataset_completo)}")
print(dataset_completo.groupby('Anno').size().to_string())

selected_row = dataset_completo[dataset_completo['Comune_Key'] == 'ASSISI']
display(selected_row[['Comune_Key', 'Anno', 'Popolazione Totale', 'Numero contribuenti']])
display(selected_row)

Anni disponibili in umbria_income: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Anni disponibili in istat_aggreg: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Anni popolazione scartati: [2026]
Anni reddito scartati:    []
Righe totali dopo il merge: 460
Anno
2021    92
2022    92
2023    92
2024    92
2025    92


,Comune_Key,Anno,Popolazione Totale,Numero contribuenti
0,ASSISI,2021,28004.0,20002
1,ASSISI,2022,27880.0,20738
2,ASSISI,2023,27671.0,20908
3,ASSISI,2024,27507.0,21192
4,ASSISI,2025,27468.0,20543


,Anno di imposta,Codice catastale,Codice Istat Comune,Denominazione Comune,Sigla Provincia,Regione,Codice Istat Regione,Numero contribuenti,Reddito da fabbricati - Frequenza,Reddito da fabbricati - Ammontare in euro,Reddito da lavoro dipendente e assimilati - Frequenza,Reddito da lavoro dipendente e assimilati - Ammontare in euro,Reddito da pensione - Frequenza,Reddito da pensione - Ammontare in euro,Reddito da lavoro autonomo (comprensivo dei valori nulli) - Frequenza,Reddito da lavoro autonomo (comprensivo dei valori nulli) - Ammontare in euro,Reddito di spettanza dell'imprenditore in contabilita' ordinaria (comprensivo dei valori nulli) - Frequenza,Reddito di spettanza dell'imprenditore in contabilita' ordinaria (comprensivo dei valori nulli) - Ammontare in euro,Reddito di spettanza dell'imprenditore in contabilita' semplificata (comprensivo dei valori nulli) - Frequenza,Reddito di spettanza dell'imprenditore in contabilita' semplificata (comprensivo dei valori nulli) - Ammontare in euro,Reddito da partecipazione (comprensivo dei valori nulli) - Frequenza,Reddito da partecipazione (comprensivo dei valori nulli) - Ammontare in euro,Reddito imponibile - Frequenza,Reddito imponibile - Ammontare in euro,Imposta netta - Frequenza,Imposta netta - Ammontare in euro,Bonus spettante - Frequenza,Bonus spettante - Ammontare in euro,Reddito imponibile addizionale - Frequenza,Reddito imponibile addizionale - Ammontare in euro,Addizionale regionale dovuta - Frequenza,Addizionale regionale dovuta - Ammontare in euro,Addizionale comunale dovuta - Frequenza,Addizionale comunale dovuta - Ammontare in euro,Reddito complessivo minore o uguale a zero euro - Frequenza,Reddito complessivo minore o uguale a zero euro - Ammontare in euro,Reddito complessivo da 0 a 10000 euro - Frequenza,Reddito complessivo da 0 a 10000 euro - Ammontare in euro,Reddito complessivo da 10000 a 15000 euro - Frequenza,Reddito complessivo da 10000 a 15000 euro - Ammontare in euro,Reddito complessivo da 15000 a 26000 euro - Frequenza,Reddito complessivo da 15000 a 26000 euro - Ammontare in euro,Reddito complessivo da 26000 a 55000 euro - Frequenza,Reddito complessivo da 26000 a 55000 euro - Ammontare in euro,Reddito complessivo da 55000 a 75000 euro - Frequenza,Reddito complessivo da 55000 a 75000 euro - Ammontare in euro,Reddito complessivo da 75000 a 120000 euro - Frequenza,Reddito complessivo da 75000 a 120000 euro - Ammontare in euro,Reddito complessivo oltre 120000 euro - Frequenza,Reddito complessivo oltre 120000 euro - Ammontare in euro,Reddito complessivo - Ammontare in euro_CALCOLATO,Trattamento spettante - Frequenza,Trattamento spettante - Ammontare in euro,Reddito complessivo - Frequenza,Reddito complessivo - Ammontare in euro,,Codice_Comune,Anno,Comune_Key,Popolazione Totale
0,2020,A475,054001,ASSISI,PG,Umbria,10,20002,8652,11674585,10344,201574854,7474,127178328,232.0,9617734.0,130.0,1979570.0,579.0,8732974.0,1440.0,17453575.0,19011,367406009,15023,67838294,5988.0,2611625.0,14754,343269746,14503,4839912,607.0,113855.0,6.0,-15431.0,5529,26472329,3003.0,37488214.0,6479,130361447,3838,130307862,311.0,19780234.0,226.0,20886211.0,116.0,27054298.0,392335164.0,NaN,NaN,NaN,NaN,NaN,54001,2021,ASSISI,28004.0
1,2021,A475,054001,ASSISI,PG,Umbria,10,20738,8744,11716155,11140,218213899,7497,128981913,241.0,11175381.0,114.0,2908416.0,563.0,9985391.0,1389.0,20723150.0,19846,392891544,15715,74689077,NaN,NaN,15432,370165458,15183,5258504,670.0,136272.0,0.0,0.0,5722,24614571,2979.0,37266458.0,6666,135458239,4143,139989321,343.0,21732804.0,272.0,25035033.0,135.0,34775523.0,418871949.0,6810.0,7392001.0,NaN,NaN,NaN,54001,2022,ASSISI,27880.0
2,2022,A475,054001,ASSISI,PG,Umbria,10,20908,8841,12504416,11454,231898641,7471,134757608,236.0,12235601.0,109.0,3098658.0,559.0,12315078.0,1354.0,23719016.0,20034,420007118,16103,76206828,NaN,NaN,15838,398091861,15634,5704291,668.0,155112.0,0.0,0.0,5318,22096398,2754.0,34612365.0,6858,140293966,4653,158431204,390.0,24903032.0,293.0,26968874.0,

**Feature Distinction: Total Population vs. Taxpayers**
When comparing the demographic and economic features, there is a consistent gap between `Popolazione Totale` and `Numero contribuenti`. The total population counts every registered resident in the municipality. The taxpayer metric only accounts for individuals who filed an income tax return (IRPEF). This structural gap filters out children, students, and adults without independent declarable income, making the taxpayer count a useful proxy for the active workforce and income-generating base of the town.

In [22]:
#From OMI Data, prices in €/m² for each macro-area (from OMI tables 11 and 15)
# OMI prices in €/m² per macro-area and year (tables 11 and 15 of each OMI report).
omi_2025 = {
    # Perugia Macro-areas (Table 11)
    'ALTO TEVERE': 782,
    'ASSISIATE': 982,
    'COLLI DEL TRASIMENO': 802,
    'EUGUBINO-GUALDESE': 705,
    'FOLIGNATE': 886,
    'MEDIA VALLE DEL TEVERE': 825,
    'MONTI MARTANI': 556,
    'PERUGINO': 940,
    'SPOLETINO': 958,
    'PERUGIA': 1075, # Solo capoluogo
    
    # Terni Macro-areas (Table 15)
    'AMERINO': 782,
    'NARNESE': 832,
    'ORVIETANO': 898,
    'VAL NERINA': 743,
    'TERNI': 895 # Solo capoluogo
}

omi_prices_by_year = {
    2021: {
         # Perugia Macro-areas (Table 11)
            'ALTO TEVERE': 799,
            'ASSISIATE': 993,
            'COLLI DEL TRASIMENO': 811,
            'EUGUBINO-GUALDESE': 715,
            'FOLIGNATE': 910,
            'MEDIA VALLE DEL TEVERE': 855,
            'MONTI MARTANI': 558,
            'PERUGINO': 927,
            'SPOLETINO': 1002,
            'PERUGIA': 1148, # Solo capoluogo
            
            # Terni Macro-areas (Table 15)
            'AMERINO': 807,
            'NARNESE': 882,
            'ORVIETANO': 1026,
            'VAL NERINA': 761,
            'TERNI': 983 # Solo capoluogo

    },   
    2022: {
         # Perugia Macro-areas (Table 11)
            'ALTO TEVERE': 800,
            'ASSISIATE': 996,
            'COLLI DEL TRASIMENO': 811,
            'EUGUBINO-GUALDESE': 712,
            'FOLIGNATE': 905,
            'MEDIA VALLE DEL TEVERE': 848,
            'MONTI MARTANI': 557,
            'PERUGINO': 929,
            'SPOLETINO': 991,
            'PERUGIA': 1119, # Solo capoluogo
            
            # Terni Macro-areas (Table 15)
            'AMERINO': 758,
            'NARNESE': 786,
            'ORVIETANO': 853,
            'VAL NERINA': 700,
            'TERNI': 787 # Solo capoluogo
    },   
    2023: {
         # Perugia Macro-areas (Table 11)
            'ALTO TEVERE': 790,
            'ASSISIATE': 992,
            'COLLI DEL TRASIMENO': 802,
            'EUGUBINO-GUALDESE': 705,
            'FOLIGNATE': 899,
            'MEDIA VALLE DEL TEVERE': 838,
            'MONTI MARTANI': 556,
            'PERUGINO': 926,
            'SPOLETINO': 973,
            'PERUGIA': 1099, # Solo capoluogo
            
            # Terni Macro-areas (Table 15)
            'AMERINO': 746,
            'NARNESE': 776,
            'ORVIETANO': 853,
            'VAL NERINA': 691,
            'TERNI': 786 # Solo capoluogo
    },   
    2024: {
         # Perugia Macro-areas (Table 11)
            'ALTO TEVERE': 786,
            'ASSISIATE': 987,
            'COLLI DEL TRASIMENO': 801,
            'EUGUBINO-GUALDESE': 711,
            'FOLIGNATE': 890,
            'MEDIA VALLE DEL TEVERE': 830,
            'MONTI MARTANI': 556,
            'PERUGINO': 938,
            'SPOLETINO': 966,
            'PERUGIA': 1078, # Solo capoluogo
            
            # Terni Macro-areas (Table 15)
            'AMERINO': 743,
            'NARNESE': 788,
            'ORVIETANO': 870,
            'VAL NERINA': 689,
            'TERNI': 809 # Solo capoluogo
    },   # TODO
    2025: omi_2025,
}

comune_to_macroarea = {
    # --- PROVINCIA DI PERUGIA ---
    # Alto Tevere
    'CITERNA': 'ALTO TEVERE', 'CITTA\' DI CASTELLO': 'ALTO TEVERE', 'LISCIANO NICCONE': 'ALTO TEVERE', 
    'MONTE SANTA MARIA TIBERINA': 'ALTO TEVERE', 'MONTONE': 'ALTO TEVERE', 'PIETRALUNGA': 'ALTO TEVERE', 
    'SAN GIUSTINO': 'ALTO TEVERE', 'UMBERTIDE': 'ALTO TEVERE',
    
    # Assisiate
    'ASSISI': 'ASSISIATE', 'BASTIA UMBRA': 'ASSISIATE', 'BETTONA': 'ASSISIATE', 'CANNARA': 'ASSISIATE',
    
    # Colli del Trasimeno
    'CASTIGLIONE DEL LAGO': 'COLLI DEL TRASIMENO', 'CITTA\' DELLA PIEVE': 'COLLI DEL TRASIMENO', 
    'MAGIONE': 'COLLI DEL TRASIMENO', 'PACIANO': 'COLLI DEL TRASIMENO', 'PANICALE': 'COLLI DEL TRASIMENO', 
    'PASSIGNANO SUL TRASIMENO': 'COLLI DEL TRASIMENO', 'PIEGARO': 'COLLI DEL TRASIMENO', 
    'TUORO SUL TRASIMENO': 'COLLI DEL TRASIMENO',
    
    # Eugubino-Gualdese
    'COSTACCIARO': 'EUGUBINO-GUALDESE', 'FOSSATO DI VICO': 'EUGUBINO-GUALDESE', 'GUALDO TADINO': 'EUGUBINO-GUALDESE', 
    'GUBBIO': 'EUGUBINO-GUALDESE', 'SCHEGGIA E PASCELUPO': 'EUGUBINO-GUALDESE', 'SIGILLO': 'EUGUBINO-GUALDESE', 
    'VALFABBRICA': 'EUGUBINO-GUALDESE',
    
    # Folignate
    'BEVAGNA': 'FOLIGNATE', 'FOLIGNO': 'FOLIGNATE', 'MONTEFALCO': 'FOLIGNATE', 'NOCERA UMBRA': 'FOLIGNATE', 
    'SPELLO': 'FOLIGNATE', 'TREVI': 'FOLIGNATE', 'VALTOPINA': 'FOLIGNATE',
    
    # Media Valle del Tevere
    'COLLAZZONE': 'MEDIA VALLE DEL TEVERE', 'DERUTA': 'MEDIA VALLE DEL TEVERE', 'FRATTA TODINA': 'MEDIA VALLE DEL TEVERE', 
    'MARSCIANO': 'MEDIA VALLE DEL TEVERE', 'MONTE CASTELLO DI VIBIO': 'MEDIA VALLE DEL TEVERE', 'TODI': 'MEDIA VALLE DEL TEVERE',
    
    # Monti Martani
    'CASTEL RITALDI': 'MONTI MARTANI', 'GIANO DELL\'UMBRIA': 'MONTI MARTANI', 'MASSA MARTANA': 'MONTI MARTANI',
    
    # Perugino
    'CORCIANO': 'PERUGINO', 'PERUGIA': 'PERUGIA', 'TORGIANO': 'PERUGINO',
    
    # Spoletino
    'CAMPELLO SUL CLITUNNO': 'SPOLETINO', 'GUALDO CATTANEO': 'SPOLETINO', 'SPOLETO': 'SPOLETINO',
    
    # Val Nerina (Perugia)
    'CASCIA': 'VAL NERINA', 'CERRETO DI SPOLETO': 'VAL NERINA', 'MONTELEONE DI SPOLETO': 'VAL NERINA', 
    'NORCIA': 'VAL NERINA', 'POGGIODOMO': 'VAL NERINA', 'PRECI': 'VAL NERINA', 'SANT\'ANATOLIA DI NARCO': 'VAL NERINA', 
    'SCHEGGINO': 'VAL NERINA', 'SELLANO': 'VAL NERINA', 'VALLO DI NERA': 'VAL NERINA',

    # --- PROVINCIA DI TERNI ---
    # Amerino
    'ACQUASPARTA': 'AMERINO', 'ALVIANO': 'AMERINO', 'AMELIA': 'AMERINO', 'ATTIGLIANO': 'AMERINO', 
    'AVIGLIANO UMBRO': 'AMERINO', 'GIOVE': 'AMERINO', 'GUARDEA': 'AMERINO', 'LUGNANO IN TEVERINA': 'AMERINO', 
    'MONTECASTRILLI': 'AMERINO', 'MONTECCHIO': 'AMERINO', 'PENNA IN TEVERINA': 'AMERINO',
    
    # Narnese
    'CALVI DELL\'UMBRIA': 'NARNESE', 'NARNI': 'NARNESE', 'OTRICOLI': 'NARNESE', 
    'SAN GEMINI': 'NARNESE', 'STRONCONE': 'NARNESE',
    
    # Orvietano
    'ALLERONA': 'ORVIETANO', 'BASCHI': 'ORVIETANO', 'CASTEL GIORGIO': 'ORVIETANO', 'CASTEL VISCARDO': 'ORVIETANO', 
    'FABRO': 'ORVIETANO', 'FICULLE': 'ORVIETANO', 'MONTEGABBIONE': 'ORVIETANO', 'MONTELEONE D\'ORVIETO': 'ORVIETANO', 
    'ORVIETO': 'ORVIETANO', 'PARRANO': 'ORVIETANO', 'PORANO': 'ORVIETANO', 'SAN VENANZO': 'ORVIETANO',
    
    # Terni Capoluogo
    'TERNI': 'TERNI',
    
    # Valnerina (Terni)
    'ARRONE': 'VAL NERINA', 'FERENTILLO': 'VAL NERINA', 'MONTEFRANCO': 'VAL NERINA', 'POLINO': 'VAL NERINA'
}

# Long table (Macroarea_OMI, Anno, Prezzo_Medio_Mq) and merge on macro-area + year
omi_long = (
    pd.DataFrame(omi_prices_by_year).rename_axis('Macroarea_OMI').reset_index()
    .melt(id_vars='Macroarea_OMI', var_name='Anno', value_name='Prezzo_Medio_Mq')
)
omi_long['Anno'] = omi_long['Anno'].astype(int)

dataset_completo['Macroarea_OMI'] = dataset_completo['Comune_Key'].map(comune_to_macroarea)
dataset_completo = (
    dataset_completo.drop(columns='Prezzo_Medio_Mq', errors='ignore')
    .merge(omi_long, on=['Macroarea_OMI', 'Anno'], how='left', validate='many_to_one')
)

display(dataset_completo[['Comune_Key', 'Anno', 'Macroarea_OMI', 'Prezzo_Medio_Mq']].head(10))

# Sanity check: macro-area/year combinations without a price (empty years above, or keys like 'VAL NERINA' missing)
no_price = (dataset_completo[dataset_completo['Prezzo_Medio_Mq'].isna()]
            .groupby(['Anno', 'Macroarea_OMI']).size())
print("Righe senza prezzo OMI:")
print(no_price.to_string() if len(no_price) else "none")

display(dataset_completo.head(10))

,Comune_Key,Anno,Macroarea_OMI,Prezzo_Medio_Mq
0,ASSISI,2021,ASSISIATE,993
1,ASSISI,2022,ASSISIATE,996
2,ASSISI,2023,ASSISIATE,992
3,ASSISI,2024,ASSISIATE,987
4,ASSISI,2025,ASSISIATE,982
5,BASTIA UMBRA,2021,ASSISIATE,993
6,BASTIA UMBRA,2022,ASSISIATE,996
7,BASTIA UMBRA,2023,ASSISIATE,992
8,BASTIA UMBRA,2024,ASSISIATE,987
9,BASTIA UMBRA,2025,ASSISIATE,982


Righe senza prezzo OMI:
none


,Anno di imposta,Codice catastale,Codice Istat Comune,Denominazione Comune,Sigla Provincia,Regione,Codice Istat Regione,Numero contribuenti,Reddito da fabbricati - Frequenza,Reddito da fabbricati - Ammontare in euro,Reddito da lavoro dipendente e assimilati - Frequenza,Reddito da lavoro dipendente e assimilati - Ammontare in euro,Reddito da pensione - Frequenza,Reddito da pensione - Ammontare in euro,Reddito da lavoro autonomo (comprensivo dei valori nulli) - Frequenza,Reddito da lavoro autonomo (comprensivo dei valori nulli) - Ammontare in euro,Reddito di spettanza dell'imprenditore in contabilita' ordinaria (comprensivo dei valori nulli) - Frequenza,Reddito di spettanza dell'imprenditore in contabilita' ordinaria (comprensivo dei valori nulli) - Ammontare in euro,Reddito di spettanza dell'imprenditore in contabilita' semplificata (comprensivo dei valori nulli) - Frequenza,Reddito di spettanza dell'imprenditore in contabilita' semplificata (comprensivo dei valori nulli) - Ammontare in euro,Reddito da partecipazione (comprensivo dei valori nulli) - Frequenza,Reddito da partecipazione (comprensivo dei valori nulli) - Ammontare in euro,Reddito imponibile - Frequenza,Reddito imponibile - Ammontare in euro,Imposta netta - Frequenza,Imposta netta - Ammontare in euro,Bonus spettante - Frequenza,Bonus spettante - Ammontare in euro,Reddito imponibile addizionale - Frequenza,Reddito imponibile addizionale - Ammontare in euro,Addizionale regionale dovuta - Frequenza,Addizionale regionale dovuta - Ammontare in euro,Addizionale comunale dovuta - Frequenza,Addizionale comunale dovuta - Ammontare in euro,Reddito complessivo minore o uguale a zero euro - Frequenza,Reddito complessivo minore o uguale a zero euro - Ammontare in euro,Reddito complessivo da 0 a 10000 euro - Frequenza,Reddito complessivo da 0 a 10000 euro - Ammontare in euro,Reddito complessivo da 10000 a 15000 euro - Frequenza,Reddito complessivo da 10000 a 15000 euro - Ammontare in euro,Reddito complessivo da 15000 a 26000 euro - Frequenza,Reddito complessivo da 15000 a 26000 euro - Ammontare in euro,Reddito complessivo da 26000 a 55000 euro - Frequenza,Reddito complessivo da 26000 a 55000 euro - Ammontare in euro,Reddito complessivo da 55000 a 75000 euro - Frequenza,Reddito complessivo da 55000 a 75000 euro - Ammontare in euro,Reddito complessivo da 75000 a 120000 euro - Frequenza,Reddito complessivo da 75000 a 120000 euro - Ammontare in euro,Reddito complessivo oltre 120000 euro - Frequenza,Reddito complessivo oltre 120000 euro - Ammontare in euro,Reddito complessivo - Ammontare in euro_CALCOLATO,Trattamento spettante - Frequenza,Trattamento spettante - Ammontare in euro,Reddito complessivo - Frequenza,Reddito complessivo - Ammontare in euro,,Codice_Comune,Anno,Comune_Key,Popolazione Totale,Macroarea_OMI,Prezzo_Medio_Mq
0,2020,A475,054001,ASSISI,PG,Umbria,10,20002,8652,11674585,10344,201574854,7474,127178328,232.0,9617734.0,130.0,1979570.0,579.0,8732974.0,1440.0,17453575.0,19011,367406009,15023,67838294,5988.0,2611625.0,14754,343269746,14503,4839912,607.0,113855.0,6.0,-15431.0,5529,26472329,3003.0,37488214.0,6479,130361447,3838,130307862,311.0,19780234.0,226.0,20886211.0,116.0,27054298.0,392335164.0,NaN,NaN,NaN,NaN,NaN,54001,2021,ASSISI,28004.0,ASSISIATE,993
1,2021,A475,054001,ASSISI,PG,Umbria,10,20738,8744,11716155,11140,218213899,7497,128981913,241.0,11175381.0,114.0,2908416.0,563.0,9985391.0,1389.0,20723150.0,19846,392891544,15715,74689077,NaN,NaN,15432,370165458,15183,5258504,670.0,136272.0,0.0,0.0,5722,24614571,2979.0,37266458.0,6666,135458239,4143,139989321,343.0,21732804.0,272.0,25035033.0,135.0,34775523.0,418871949.0,6810.0,7392001.0,NaN,NaN,NaN,54001,2022,ASSISI,27880.0,ASSISIATE,996
2,2022,A475,054001,ASSISI,PG,Umbria,10,20908,8841,12504416,11454,231898641,7471,134757608,236.0,12235601.0,109.0,3098658.0,559.0,12315078.0,1354.0,23719016.0,20034,420007118,16103,76206828,NaN,NaN,15838,398091861,15634,5704291,668.0,155112.0,0.0,0.0,5318,22096398,2754.0,34612365.0,6858,1

In [26]:
import numpy as np
import pandas as pd

# 1. Ricerca flessibile per la colonna del reddito totale e le fasce
col_reddito = 'Reddito complessivo - Ammontare in euro_CALCOLATO'
if col_reddito not in dataset_completo.columns:
    income_col = [c for c in dataset_completo.columns if 'Reddito complessivo' in c and 'Ammontare' in c]
    if not income_col:
        raise ValueError("Income total column not found!")
    col_reddito = income_col[0]

print(f"Using income column for GDP per capita: {col_reddito}")

# Isoliamo dinamicamente tutte le colonne relative alle fasce di reddito (escludendo il totale)
bracket_cols = [c for c in dataset_completo.columns if ('Ammontare in euro' in c or 'Frequenza' in c) and c != col_reddito]

# 2. Conversione forzata a numerico per tutte le metriche
core_cols = [col_reddito, 'Popolazione Totale', 'Numero contribuenti']
all_numeric_cols = core_cols + bracket_cols

for c in all_numeric_cols:
    dataset_completo[c] = pd.to_numeric(dataset_completo[c], errors='coerce')

# 3. Imputazione con la mediana per anno tramite Pandas (Previene l'IndexError)
for col in all_numeric_cols:
    # Calcola la mediana per anno e riempie i NaN
    dataset_completo[col] = dataset_completo.groupby('Anno')[col].transform(lambda x: x.fillna(x.median()))
    
    # Se per un certo anno la colonna era 100% NaN, la mediana stessa è NaN. 
    # Per queste casistiche limite (es. fasce assenti in un'annata vecchia), riempiamo con 0
    dataset_completo[col] = dataset_completo[col].fillna(0)

# 4. Calcolo delle metriche Pro Capite (equivalente al GDP pro capite)
dataset_completo['Reddito_Pro_Capite'] = dataset_completo[col_reddito] / dataset_completo['Popolazione Totale']
dataset_completo['Reddito_Medio_Contribuente'] = dataset_completo[col_reddito] / dataset_completo['Numero contribuenti']

# 5. Variazione Year-over-Year (YoY) per comune
dataset_completo = dataset_completo.sort_values(['Codice_Comune', 'Anno']).reset_index(drop=True)

# Controllo sicurezza sui salti temporali
gap_check = dataset_completo.groupby('Codice_Comune')['Anno'].diff().dropna()
assert (gap_check == 1).all(), "Gap negli anni per qualche comune!"

# Definiamo le metriche su cui vogliamo calcolare la variazione percentuale (includendo le metriche aggregate)
yoy_cols = ['Popolazione Totale', 'Numero contribuenti', col_reddito,
            'Reddito_Pro_Capite', 'Reddito_Medio_Contribuente', 'Prezzo_Medio_Mq']

grp = dataset_completo.groupby('Codice_Comune')
for c in yoy_cols:
    dataset_completo[f'{c}_diff'] = grp[c].diff()                                # Delta assoluto
    dataset_completo[f'{c}_yoy_pct'] = grp[c].pct_change(fill_method=None) * 100 # Variazione %

# 6. Selezione delle colonne e Display
comune_col_name = 'Comune_Key' if 'Comune_Key' in dataset_completo.columns else 'Codice_Comune'

# Selezioniamo alcune fasce a titolo esemplificativo per assicurarci che siano presenti
sample_brackets = [b for b in bracket_cols if 'Reddito complessivo da' in b][:2] 

cols_view = [comune_col_name, 'Anno', 'Popolazione Totale', 
             'Reddito_Pro_Capite', 'Reddito_Pro_Capite_yoy_pct',
             'Prezzo_Medio_Mq'] + sample_brackets

display(dataset_completo[cols_view].head(12))
print(dataset_completo.shape)

Using income column for GDP per capita: Reddito complessivo - Ammontare in euro_CALCOLATO


,Comune_Key,Anno,Popolazione Totale,Reddito_Pro_Capite,Reddito_Pro_Capite_yoy_pct,Prezzo_Medio_Mq,Reddito complessivo da 0 a 10000 euro - Frequenza,Reddito complessivo da 0 a 10000 euro - Ammontare in euro
0,ASSISI,2021,28004.0,14009.968719,NaN,993,5529,26472329
1,ASSISI,2022,27880.0,15024.101471,7.238651,996,5722,24614571
2,ASSISI,2023,27671.0,16212.909653,7.912674,992,5318,22096398
3,ASSISI,2024,27507.0,33198.013415,104.762835,987,5013,20243157
4,ASSISI,2025,27468.0,34816.735692,4.875961,982,4000,18321829
5,BASTIA UMBRA,2021,21379.0,13773.733991,NaN,993,3867,18330888
6,BASTIA UMBRA,2022,21256.0,14710.926044,6.804197,996,4301,17543497
7,BASTIA UMBRA,2023,21267.0,15860.178963,7.812240,992,4106,16475588
8,BASTIA UMBRA,2024,21292.0,34113.383900,115.088266,987,4057,15412059
9,BASTIA UMBRA,2025,21321.0,34696.851930,1.710379,982,3086,14123416


(460, 76)


In [28]:
# 1. Identifichiamo i core predictors che vogliamo assolutamente tenere
core_features = [
    'Comune_Key', 'Anno', 
    'Popolazione Totale', 'Popolazione Totale_yoy_pct',
    'Numero contribuenti', 
    'Reddito_Pro_Capite', 'Reddito_Pro_Capite_yoy_pct',
    'Reddito_Medio_Contribuente', 
    'Prezzo_Medio_Mq', 'Prezzo_Medio_Mq_yoy_pct'
]

# 2. Selezioniamo SOLO le colonne di Ammontare delle fasce di reddito (ignorando le Frequenze)
bracket_amount_cols = [c for c in dataset_completo.columns if 'Reddito complessivo' in c and 'Ammontare in euro' in c and '-' in c and 'CALCOLATO' not in c]

# 3. Creiamo le "Ratio Features" (Percentuale di ricchezza per fascia)
# Questo cattura la distribuzione della ricchezza neutralizzando l'effetto della grandezza del comune
for col in bracket_amount_cols:
    # Estraiamo un nome pulito per la nuova feature (es. "Pct_Reddito_da_0_a_10000")
    clean_name = col.replace('Reddito complessivo ', 'Pct_').replace(' - Ammontare in euro', '').replace(' ', '_')
    # Calcoliamo la percentuale rispetto al reddito totale calcolato in precedenza
    dataset_completo[clean_name] = (dataset_completo[col] / dataset_completo[col_reddito]) * 100
    dataset_completo[clean_name] = dataset_completo[clean_name].fillna(0)
    
    # Aggiungiamo la nuova ratio feature alla lista delle colonne finali
    core_features.append(clean_name)

# 4. Creiamo il Dataset finale ridotto
dataset_modello = dataset_completo[core_features].copy()

#remove unwanted column
dataset_modello = dataset_modello.drop("Pct_-_Ammontare_in_euro", axis=1)

print(f"Dim original dataset: {dataset_completo.shape}")
print(f"Dim reduced dataset: {dataset_modello.shape}")

# Anteprima delle nuove features proporzionali
display(dataset_modello.head())

Dim original dataset: (460, 85)
Dim reduced dataset: (460, 18)


,Comune_Key,Anno,Popolazione Totale,Popolazione Totale_yoy_pct,Numero contribuenti,Reddito_Pro_Capite,Reddito_Pro_Capite_yoy_pct,Reddito_Medio_Contribuente,Prezzo_Medio_Mq,Prezzo_Medio_Mq_yoy_pct,Pct_minore_o_uguale_a_zero_euro,Pct_da_0_a_10000_euro,Pct_da_10000_a_15000_euro,Pct_da_15000_a_26000_euro,Pct_da_26000_a_55000_euro,Pct_da_55000_a_75000_euro,Pct_da_75000_a_120000_euro,Pct_oltre_120000_euro
0,ASSISI,2021,28004.0,NaN,20002,14009.968719,NaN,19614.796720,993,NaN,-0.003933,6.747376,9.555150,33.227062,33.213404,5.041667,5.323563,6.895711
1,ASSISI,2022,27880.0,-0.442794,20738,15024.101471,7.238651,20198.280885,996,0.302115,0.000000,5.876395,8.896862,32.338818,33.420553,5.188412,5.976775,8.302185
2,ASSISI,2023,27671.0,-0.749641,20908,16212.909653,7.912674,21457.213650,992,-0.401606,0.000000,4.925334,7.715169,31.271821,35.314650,5.550938,6.011419,9.210668
3,ASSISI,2024,27507.0,-0.592678,21192,33198.013415,104.762835,43090.683041,987,-0.504032,0.000000,2.216782,3.450884,15.243439,20.254089,3.114713,3.357632,0.000000
4,ASSISI,2025,27468.0,-0.141782,20543,34816.735692,4.875961,46553.380519,982,-0.506586,0.000000,1.915816,2.999751,14.166392,22.024614,3.279276,3.527401,0.000000


In [ ]:
#save final dataset to CSV

dataset_modello.to_csv("umbria_housing_model_ready.csv", index=False)